In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.dates as mdates

# Load fire obs

In [ ]:
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region_w_country.shp")
fire_obs["start_date"] = pd.to_datetime(fire_obs["start_date"])

# Load study area

In [ ]:
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")
ctr_list = list(study_area['ISO3_CODE'].values)

In [ ]:
len(ctr_list)  #45 countries

# Time Series (All Seasons)

In [ ]:
timestep = pd.date_range(start = "2001-01-01", end = "2020-12-31")

In [ ]:
fire_time_series_all = pd.DataFrame({"Time": timestep})

for ctr in ctr_list:
    
    # country-level fire time series
    fire_time_series_ctr = pd.DataFrame(fire_obs.loc[fire_obs["country"] == ctr, "start_date"].value_counts()).reset_index()
    fire_time_series_ctr["start_date"] = pd.to_datetime(fire_time_series_ctr["start_date"])
    fire_time_series_ctr = fire_time_series_ctr.rename(columns = {"start_date": "Time", "count": ctr})

    # merge
    fire_time_series_all = pd.merge(fire_time_series_all, fire_time_series_ctr, how = "left", on = "Time")

fire_time_series_all = fire_time_series_all.fillna(0)   #NAN means no fire happening, fill with 0

# convert to int
fire_time_series_all[fire_time_series_all.select_dtypes(include = ['number']).columns] = fire_time_series_all.select_dtypes(include = ['number']).astype(int)

In [ ]:
# check total fire number (correct)
fire_time_series_all[fire_time_series_all.select_dtypes(include = ['number']).columns].sum()

In [ ]:
# quick visualization
fig, ax = plt.subplots(1, 1, figsize = (15, 2.5))

ctr = ctr_list[0]

for date, enum in zip(fire_time_series_all["Time"], fire_time_series_all[ctr]):  #date, event number
    ax.plot([date, date], [0, enum], color = "red", linewidth = 0.8)  #[x1, x2], [y1, y2]
    
    
ax.set_ylim(0, fire_time_series_all[ctr].max() + round(fire_time_series_all[ctr].max() * 0.1, 0))
ax.set_title(ctr)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth = [3, 6, 9, 12]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_xlim(datetime(2001, 1, 1), datetime(2020, 12, 31))
ax.tick_params(axis="x", rotation=45, labelsize = 6)

plt.subplots_adjust(hspace = 1)
plt.show()

In [ ]:
#fire_time_series_all.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Time_Series_Extraction/fire_time_series_country-level_all_seasons.csv", index = False)

# Time Series (MAM, JJA, SON, DJF)

In [ ]:
#MAM
fire_time_series_MAM = fire_time_series_all[fire_time_series_all["Time"].dt.month.isin([3,4,5])]
#fire_time_series_MAM.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Time_Series_Extraction/fire_time_series_country-level_MAM.csv", index = False)

In [ ]:
#JJA
fire_time_series_JJA = fire_time_series_all[fire_time_series_all["Time"].dt.month.isin([6,7,8])]
#fire_time_series_JJA.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Time_Series_Extraction/fire_time_series_country-level_JJA.csv", index = False)

In [ ]:
#SON
fire_time_series_SON = fire_time_series_all[fire_time_series_all["Time"].dt.month.isin([9,10,11])]
#fire_time_series_SON.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Time_Series_Extraction/fire_time_series_country-level_SON.csv", index = False)

In [ ]:
#DJF
fire_time_series_DJF = fire_time_series_all[fire_time_series_all["Time"].dt.month.isin([12,1,2])]
#fire_time_series_DJF.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Time_Series_Extraction/fire_time_series_country-level_DJF.csv", index = False)